# 화면 이벤트에서 수정 요청과 인수 시험까지

이름이 같은 할 일이 두 개 있다. 학생이 두 번째 행을 바꿨을 때 선택이 끝까지 유지되는지 화면·요청·결과에서 확인한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 화면의 순번과 데이터의 식별자

화면을 정렬하거나 행을 삭제하면 순번은 바뀐다. 배열의 두 번째 위치나 같은 제목 문자열로 수정할 대상을 보내면 사용자가 누른 행과 서버가 고친 행이 달라질 수 있다.

요청에는 영속 ID와 사용자가 원하는 최종 상태를 담는다. 완료 버튼을 누른 사건만 전달해 상태를 뒤집게 하면 같은 요청이 두 번 도착했을 때 미완료로 돌아온다.

먼저 화면과 API 사이에서 주고받을 데이터 예를 노트북으로 확인한다. 이후 같은 입력을 실제 JavaScript 화면에서 만들어 네트워크 요청 본문과 비교한다.

## 두 번 도착한 완료 요청

노트북에서는 API가 받는 JSON과 같은 구조의 딕셔너리로 상태 변경 규칙을 시험한다.

In [1]:
items = {10:{"title":"정리","done":False}, 21:{"title":"정리","done":False}}
def apply_patch(items, request):
    row = items.get(request["id"])
    if row is None:
        return "찾을 수 없음"
    row["done"] = request["done"]
    return "저장"

request = {"id":21, "done":True}
apply_patch(items, request)
apply_patch(items, request)
print([key for key, task in items.items() if task["done"]])

[21]


21번만 완료 상태다. 상태를 단순히 반전하는 코드였다면 두 번째 요청 뒤에 다시 빈 목록이 된다.

### 이미 지운 행을 수정하는 요청

사용자가 버튼을 누른 사이 다른 화면에서 21번을 삭제했다. 남은 10번을 실수로 수정하지 않는지 확인한다.

In [2]:
items.pop(21, None)

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>print(apply_patch(items, request), items[10][&quot;done&quot;])</code></pre><pre>찾을 수 없음 False
</pre><p>없는 ID는 찾을 수 없음으로 끝나며 이름이 같은 10번은 바뀌지 않는다.</p></details>

## 사용자가 선택한 최신 요청만 화면에 표시하기

검색어가 바뀌어 요청을 다시 보냈는데 첫 요청이 늦게 도착했다. 실제 화면에는 방금 입력한 검색어의 결과가 남아야 한다.

In [4]:
pending_queries = {"r1":"안내", "r2":"안내문"}
latest = "r2"
responses = [("r2",["안내문 양식"]), ("r1",["옛 안내"])]
def current_results(latest, responses):
    result = []
    for request_id, items in responses:
        if request_id == latest:
            result = items
    return result
print(current_results(latest,responses))

['안내문 양식']


마지막에 도착한 r1을 그대로 표시하면 화면 입력과 검색 결과가 어긋난다. 완료된 응답이 어떤 요청에 속하는지 확인해야 한다.

### 결과 0건도 이번 검색 결과로 표시하기

최신 검색 결과가 빈 목록이다. 이전 결과가 더 길거나 나중에 도착했다는 이유로 화면을 덮지 않도록 입력을 바꿔 검산한다.

In [5]:
latest_empty = [("r2",[]), ("r1",["옛 안내"])]

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>print(current_results(&quot;r2&quot;, latest_empty))</code></pre><pre>[]
</pre><p>빈 목록이 유지된다. 브라우저에서는 네트워크 지연을 다르게 주어 실제 화면에서도 같은지 확인할 수 있다.</p></details>

## 다른 입력에 적용하기

Codex에 “정렬 후에도 같은 ID를 수정한다”를 수용 기준으로 전달한다. 실제 화면에 동명 항목을 두 개 만들고 두 번째 행 수정, 삭제, 새로고침을 수행한다.

첫 커밋과 수정 커밋에 같은 동작을 적용해 어떤 요청 본문과 화면 행이 바뀌었는지 설명한다.